# BoneTest — Colab GPU Training for `multiscale_cnn_rf`, `ensemble_cnn`, and `regression_multimodal`

This notebook trains the three remaining BoneTest models strictly on the frozen, deduplicated, joint-stratified RSNA split manifests (`train_manifest.csv` [$N=10,068$] and `val_manifest.csv` [$N=1,259$]), **without ever touching `test_manifest.csv` ($N=1,259$)**.

### Models Trained & Exact Output Artifacts Produced
1. **`multiscale_cnn_rf`** (`MultiscaleCNNFeatureExtractor` + `RandomForestRegressor` with 3-Fold Out-of-Fold anti-leakage feature extraction):
   - `checkpoints/multiscale_cnn_rf_best.pt`
   - `rf_models/multiscale_cnn_rf.joblib`
   - `checkpoints/multiscale_cnn_rf_history.json`
2. **`ensemble_cnn`** (`EnsembleCNN`, 3-branch ResNet-18 ensemble with learned fusion head):
   - `checkpoints/ensemble_cnn_best.pt`
   - `checkpoints/ensemble_cnn_history.json`
3. **`regression_multimodal`** (`RegressionMultimodal`, ResNet-18 + 64-d biological sex metadata encoder + deep fusion head):
   - `checkpoints/regression_multimodal_best.pt`
   - `checkpoints/regression_multimodal_history.json`

All epoch checkpoints and best artifacts are saved directly to Google Drive (`/content/drive/MyDrive/bonetest_three_models_artifacts`) so no progress is lost if Colab disconnects.

In [ ]:
# 1. Mount Google Drive & Verify GPU
import os
from pathlib import Path
import torch

from google.colab import drive
drive.mount('/content/drive')

assert torch.cuda.is_available(), 'Please enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU'
DEVICE = torch.device('cuda')
print('Using GPU:', torch.cuda.get_device_name(0))

DRIVE_ROOT = Path('/content/drive/MyDrive/bonetest_three_models_artifacts')
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
RF_DIR = DRIVE_ROOT / 'rf_models'
EPOCH_CKPT_DIR = DRIVE_ROOT / 'epoch_checkpoints'
for d in (CKPT_DIR, RF_DIR, EPOCH_CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Artifacts will be saved to:', DRIVE_ROOT)

In [ ]:
# 2. Configure Input Paths (Code, Split Manifests, normalization.json, and RSNA Images)
# Upload `bonetest_colab_bundle.zip` (containing backend/app/models, ml/data/splits, backend/checkpoints/normalization.json)
# and unzip to /content/bonetest, or point DATA_IMAGES_DIR to your RSNA boneage-training-dataset folder.
import json
import hashlib
import sys
import zipfile
import pandas as pd

BUNDLE_ZIP = Path('/content/drive/MyDrive/bonetest_colab_bundle.zip')
WORK_DIR = Path('/content/bonetest')
if BUNDLE_ZIP.exists() and not WORK_DIR.exists():
    with zipfile.ZipFile(BUNDLE_ZIP, 'r') as zf:
        zf.extractall(WORK_DIR)
    print('Extracted bundle to', WORK_DIR)

for p in [str(WORK_DIR), str(WORK_DIR / 'backend')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Set path to the folder containing the 12,611 RSNA training PNG files (e.g. 1377.png ...)
DATA_IMAGES_DIR = Path('/content/rsna-bone-age/boneage-training-dataset/boneage-training-dataset')
SPLITS_DIR = WORK_DIR / 'ml' / 'data' / 'splits'
NORM_JSON_PATH = WORK_DIR / 'backend' / 'checkpoints' / 'normalization.json'

assert SPLITS_DIR.exists(), f'Missing splits directory at {SPLITS_DIR}'
assert NORM_JSON_PATH.exists(), f'Missing normalization.json at {NORM_JSON_PATH}'
assert DATA_IMAGES_DIR.exists(), f'Missing RSNA PNG directory at {DATA_IMAGES_DIR}'

with open(NORM_JSON_PATH, 'r') as f:
    norm_stats = json.load(f)
NORM_MEAN = float(norm_stats['mean'])  # 0.4523
NORM_STD = float(norm_stats['std'])    # 0.2118
IMAGE_SIZE = 512
print(f'Loaded normalization.json: mean={NORM_MEAN}, std={NORM_STD}, image_size={IMAGE_SIZE}')

In [ ]:
# 3. Load & Strictly Verify Frozen Split Manifests (Zero Overlap, Test Set Excluded)
train_df = pd.read_csv(SPLITS_DIR / 'train_manifest.csv')
val_df = pd.read_csv(SPLITS_DIR / 'val_manifest.csv')
test_df = pd.read_csv(SPLITS_DIR / 'test_manifest.csv')

assert len(train_df) == 10068, f'Expected 10068 train rows, got {len(train_df)}'
assert len(val_df) == 1259, f'Expected 1259 val rows, got {len(val_df)}'
assert len(test_df) == 1259, f'Expected 1259 test rows, got {len(test_df)}'

s_tr, s_va, s_te = set(train_df['id']), set(val_df['id']), set(test_df['id'])
assert len(s_tr & s_va) == 0, 'Train and Val overlap!'
assert len(s_tr & s_te) == 0, 'Train and Test overlap!'
assert len(s_va & s_te) == 0, 'Val and Test overlap!'

# Delete test_df reference from training scope so test images can NEVER be used for training or tuning
del test_df, s_te
print(f'Verified disjoint manifests: Train={len(train_df)}, Val={len(val_df)} (Test set locked & excluded).')

In [ ]:
# 4. Shared Dataset, Preprocessing (Exact Match to preprocess_for_inference), and Seeding
import random
import time
import cv2
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from app.models.multiscale_cnn_rf import MultiscaleCNNFeatureExtractor, MultiscaleCNNRFWrapper
from app.models.ensemble_cnn import EnsembleCNN
from app.models.regression_multimodal import RegressionMultimodal
from ml.augmentation import TrainAugmentation

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def letterbox_resize_512(image: np.ndarray, target_size: int = 512) -> np.ndarray:
    """Exact match to backend/app/services/preprocessing.py::letterbox_resize."""
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2
    canvas[y_off : y_off + new_h, x_off : x_off + new_w] = resized
    return canvas

class ManifestBoneAgeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        fname = str(row['file_name']) if 'file_name' in row else f"{int(row['id'])}.png"
        img_path = self.image_dir / fname
        image = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f'Failed to load image: {img_path}')

        image = letterbox_resize_512(image, IMAGE_SIZE)
        if self.transform is not None:
            aug = self.transform(image=image)
            image = aug['image'] if isinstance(aug, dict) else aug

        norm = image.astype(np.float32) / 255.0
        norm = (norm - NORM_MEAN) / max(NORM_STD, 1e-6)
        tensor = torch.from_numpy(norm).unsqueeze(0).float()  # (1, 512, 512)
        bone_age = torch.tensor(float(row['boneage']), dtype=torch.float32)
        male = torch.tensor(1.0 if bool(row['male']) else 0.0, dtype=torch.float32)
        return {'image': tensor, 'bone_age': bone_age, 'male': male, 'id': str(row['id'])}

train_aug = TrainAugmentation()
train_ds = ManifestBoneAgeDataset(train_df, DATA_IMAGES_DIR, transform=train_aug)
train_eval_ds = ManifestBoneAgeDataset(train_df, DATA_IMAGES_DIR, transform=None)
val_ds = ManifestBoneAgeDataset(val_df, DATA_IMAGES_DIR, transform=None)

In [ ]:
# 5. Generic End-to-End PyTorch Training Helper (Saves Every Epoch to Drive + Best Checkpoint)
def train_pytorch_regressor(
    model: nn.Module,
    model_type: str,
    train_dataset: Dataset,
    val_dataset: Dataset,
    requires_gender: bool,
    epochs: int = 30,
    batch_size: int = 16,
    lr: float = 1e-4,
    patience: int = 6,
    seed: int = 42,
    save_best_path: Path | None = None,
    state_dict_fn=None,
) -> dict:
    set_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = model.to(DEVICE)
    criterion = nn.SmoothL1Loss(beta=1.0)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    best_mae = float('inf')
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        train_loss_sum = 0.0
        for batch in train_loader:
            imgs = batch['image'].to(DEVICE)
            targets = batch['bone_age'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=True):
                if requires_gender:
                    gender = batch['male'].unsqueeze(-1).to(DEVICE)
                    preds = model(imgs, gender)
                else:
                    preds = model(imgs)
                loss = criterion(preds, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss_sum += loss.item() * imgs.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(DEVICE)
                targets = batch['bone_age'].to(DEVICE)
                if requires_gender:
                    gender = batch['male'].unsqueeze(-1).to(DEVICE)
                    preds = model(imgs, gender)
                else:
                    preds = model(imgs)
                loss = criterion(preds, targets)
                val_loss_sum += loss.item() * imgs.size(0)
                val_preds.extend(preds.cpu().numpy().tolist())
                val_targets.extend(targets.cpu().numpy().tolist())

        val_loss = val_loss_sum / len(val_loader.dataset)
        vp = np.array(val_preds)
        vt = np.array(val_targets)
        mae = float(np.mean(np.abs(vp - vt)))
        mse = float(np.mean((vp - vt) ** 2))
        rmse = float(np.sqrt(mse))
        scheduler.step(mae)
        cur_lr = optimizer.param_groups[0]['lr']
        dt = time.time() - t0

        rec = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'mae': mae, 'mse': mse, 'rmse': rmse, 'lr': cur_lr}
        history.append(rec)
        print(f'[{model_type}] Epoch {epoch:02d}/{epochs:02d} ({dt:.1f}s) | Train: {train_loss:.4f} | Val MAE: {mae:.2f} mo | Val RMSE: {rmse:.2f} mo | LR: {cur_lr:.2e}')

        sd_to_save = state_dict_fn(model) if state_dict_fn is not None else model.state_dict()
        payload = {
            'model_state_dict': sd_to_save,
            'model_type': model_type,
            'val_mae': mae,
            'val_mse': mse,
            'val_rmse': rmse,
            'norm_mean': NORM_MEAN,
            'norm_std': NORM_STD,
            'epoch': epoch,
        }
        # Save every epoch checkpoint to Drive
        torch.save(payload, EPOCH_CKPT_DIR / f'{model_type}_epoch_{epoch:02d}.pt')

        if mae < best_mae:
            best_mae = mae
            patience_counter = 0
            if save_best_path is not None:
                torch.save(payload, save_best_path)
                print(f'  -> Saved new best checkpoint to {save_best_path} (Val MAE: {mae:.2f} mo)')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} epochs without improvement.')
                break

    return {'best_mae': best_mae, 'history': history}

In [ ]:
# 6. MODEL 1: Train `multiscale_cnn_rf` with Out-of-Fold (OOF) Anti-Leakage Protocol
#   Step 6a: 3-Fold stratified out-of-fold feature extraction on train_manifest.csv so the Random Forest
#            is trained ONLY on out-of-sample CNN features (zero in-sample CNN feature leakage).
#   Step 6b: Train final MultiscaleCNNFeatureExtractor on full train_manifest.csv (validated on val_manifest.csv)
#            and save `checkpoints/multiscale_cnn_rf_best.pt` + `rf_models/multiscale_cnn_rf.joblib`.
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold

class MultiscaleTrainRegressor(nn.Module):
    """Attaches a temporary linear regression head to train MultiscaleCNNFeatureExtractor end-to-end."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.extractor = MultiscaleCNNFeatureExtractor(pretrained=pretrained)
        self.head = nn.Linear(512, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(x)
        return self.head(feats).squeeze(-1)

@torch.no_grad()
def extract_512d_features(extractor: MultiscaleCNNFeatureExtractor, dataset: Dataset) -> tuple[np.ndarray, np.ndarray]:
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    extractor = extractor.to(DEVICE).eval()
    feats_list, ages_list = [], []
    for batch in loader:
        imgs = batch['image'].to(DEVICE)
        feats = extractor(imgs).cpu().numpy()
        feats_list.append(feats)
        ages_list.extend(batch['bone_age'].numpy().tolist())
    return np.vstack(feats_list), np.array(ages_list)

set_seed(42)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
oof_features = np.zeros((len(train_df), 512), dtype=np.float32)
oof_targets = train_df['boneage'].to_numpy(dtype=np.float32)

for fold_idx, (tr_idx, hold_idx) in enumerate(skf.split(train_df, train_df['strata']), 1):
    print(f'\n=== Multiscale CNN OOF Fold {fold_idx}/3 (train={len(tr_idx)}, out-of-fold={len(hold_idx)}) ===')
    fold_tr_ds = ManifestBoneAgeDataset(train_df.iloc[tr_idx], DATA_IMAGES_DIR, transform=train_aug)
    fold_hold_ds = ManifestBoneAgeDataset(train_df.iloc[hold_idx], DATA_IMAGES_DIR, transform=None)
    fold_model = MultiscaleTrainRegressor(pretrained=True)
    fold_ckpt = EPOCH_CKPT_DIR / f'multiscale_oof_fold{fold_idx}_best.pt'
    train_pytorch_regressor(
        model=fold_model,
        model_type=f'multiscale_oof_fold{fold_idx}',
        train_dataset=fold_tr_ds,
        val_dataset=fold_hold_ds,
        requires_gender=False,
        epochs=15,
        batch_size=16,
        lr=1e-4,
        patience=4,
        seed=42 + fold_idx,
        save_best_path=fold_ckpt,
        state_dict_fn=lambda m: m.extractor.state_dict(),
    )
    best_fold_ext = MultiscaleCNNFeatureExtractor(pretrained=False)
    best_fold_ext.load_state_dict(torch.load(fold_ckpt, map_location=DEVICE)['model_state_dict'], strict=True)
    f_hold, _ = extract_512d_features(best_fold_ext, fold_hold_ds)
    oof_features[hold_idx] = f_hold

print('\n=== Training Final MultiscaleCNNFeatureExtractor on Full train_manifest.csv ===')
final_ms_reg = MultiscaleTrainRegressor(pretrained=True)
ms_best_pt = CKPT_DIR / 'multiscale_cnn_rf_best.pt'
ms_res = train_pytorch_regressor(
    model=final_ms_reg,
    model_type='multiscale_cnn_rf',
    train_dataset=train_ds,
    val_dataset=val_ds,
    requires_gender=False,
    epochs=25,
    batch_size=16,
    lr=1e-4,
    patience=6,
    seed=42,
    save_best_path=ms_best_pt,
    state_dict_fn=lambda m: m.extractor.state_dict(),
)

print('\n=== Fitting RandomForestRegressor (200 trees) on Out-of-Fold 512-d Features ===')
rf = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(oof_features, oof_targets)

# Evaluate combined final MultiscaleCNNFeatureExtractor + OOF-trained RF on val_manifest.csv
best_ext = MultiscaleCNNFeatureExtractor(pretrained=False)
best_ext.load_state_dict(torch.load(ms_best_pt, map_location=DEVICE)['model_state_dict'], strict=True)
X_val, y_val = extract_512d_features(best_ext, val_ds)
val_preds_rf = rf.predict(X_val)
rf_val_mae = float(np.mean(np.abs(val_preds_rf - y_val)))
rf_val_rmse = float(np.sqrt(np.mean((val_preds_rf - y_val) ** 2)))
print(f'[multiscale_cnn_rf] Combined Val MAE: {rf_val_mae:.2f} months | Val RMSE: {rf_val_rmse:.2f} months')

rf.n_jobs = 1
ms_joblib_path = RF_DIR / 'multiscale_cnn_rf.joblib'
joblib.dump(rf, ms_joblib_path)
with open(CKPT_DIR / 'multiscale_cnn_rf_history.json', 'w') as f:
    json.dump({'rf_val_mae': rf_val_mae, 'rf_val_rmse': rf_val_rmse, 'cnn_history': ms_res['history']}, f, indent=2)
print('Saved:', ms_best_pt, 'and', ms_joblib_path)

In [ ]:
# 7. MODEL 2: Train `ensemble_cnn` (3-Branch ResNet-18 Ensemble)
print('\n=== Training ensemble_cnn ===')
ensemble_model = EnsembleCNN(pretrained=True, num_branches=3)
ensemble_best_pt = CKPT_DIR / 'ensemble_cnn_best.pt'
ens_res = train_pytorch_regressor(
    model=ensemble_model,
    model_type='ensemble_cnn',
    train_dataset=train_ds,
    val_dataset=val_ds,
    requires_gender=False,
    epochs=30,
    batch_size=16,
    lr=1e-4,
    patience=6,
    seed=42,
    save_best_path=ensemble_best_pt,
)
with open(CKPT_DIR / 'ensemble_cnn_history.json', 'w') as f:
    json.dump(ens_res['history'], f, indent=2)
print('Saved:', ensemble_best_pt)

In [ ]:
# 8. MODEL 3: Train `regression_multimodal` (ResNet-18 + 64-d Biological Sex Metadata Fusion)
print('\n=== Training regression_multimodal ===')
reg_multi_model = RegressionMultimodal(pretrained=True, meta_dim=1)
reg_multi_best_pt = CKPT_DIR / 'regression_multimodal_best.pt'
rm_res = train_pytorch_regressor(
    model=reg_multi_model,
    model_type='regression_multimodal',
    train_dataset=train_ds,
    val_dataset=val_ds,
    requires_gender=True,
    epochs=30,
    batch_size=16,
    lr=1e-4,
    patience=6,
    seed=42,
    save_best_path=reg_multi_best_pt,
)
with open(CKPT_DIR / 'regression_multimodal_history.json', 'w') as f:
    json.dump(rm_res['history'], f, indent=2)
print('Saved:', reg_multi_best_pt)

In [ ]:
# 9. Verify All Saved Checkpoints Load with strict=True Before Downloading
for m_name, cls in [
    ('multiscale_cnn_rf', MultiscaleCNNFeatureExtractor),
    ('ensemble_cnn', EnsembleCNN),
    ('regression_multimodal', RegressionMultimodal),
]:
    pt_file = CKPT_DIR / f'{m_name}_best.pt'
    assert pt_file.exists(), f'Missing {pt_file}'
    inst = cls(pretrained=False)
    ckpt = torch.load(pt_file, map_location='cpu', weights_only=False)
    inst.load_state_dict(ckpt['model_state_dict'], strict=True)
    print(f'[VERIFIED strict=True] {pt_file.name} ({pt_file.stat().st_size:,} bytes) | Val MAE={ckpt.get("val_mae"):.2f} mo')

rf_file = RF_DIR / 'multiscale_cnn_rf.joblib'
assert rf_file.exists(), f'Missing {rf_file}'
rf_loaded = joblib.load(rf_file)
dummy_pred = float(rf_loaded.predict(np.zeros((1, 512), dtype=np.float32))[0])
print(f'[VERIFIED joblib] {rf_file.name} ({rf_file.stat().st_size:,} bytes) | dummy prediction={dummy_pred:.2f} mo')